Our notebook used the following strategy for selecting astrophysical objects used in observation proposals. For this proposal, we sought for bright systems ($\mathrm{mag}_i<19$) with known $z_S$ and $z_L$, visible on specific dates at the SOAR telescope.

In [1]:
import io
import os

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import gridspec
from minio import Minio
from PIL import Image

matplotlib.rcParams.update(
    {
        "font.size": 20,
        # "pgf.texsystem": "pdflatex",
        "font.family": "serif",
        # "text.usetex": True,
    }
)

MINIO_ENDPOINT_URL = "nonarithmetically-undeliberating-janelle.ngrok-free.app"
ACCESS_KEY = "slcomp"
SECRET_KEY = "slcomp@data"
client = Minio(
    MINIO_ENDPOINT_URL,
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
    secure=True,
)

In [2]:
# Load the database
Database_object = client.get_object("slcomp", "Data/Database.csv").data
Database = pd.read_csv(
    io.StringIO(Database_object.decode("utf-8")), low_memory=False, dtype=object
)
Database["RA"] = Database["RA"].astype(float)
Database["DEC"] = Database["DEC"].astype(float)
Database.head()

,JNAME,Original_ID,Alternative_Name,RA,DEC,Individual_Coordinates,Grade,z_L,z_LErr,z_LType,...,mag_yErr,mag_F814W,mag_F814WErr,mag_F814WS,System_Type,Lens_Type,Source_Type,Other_Matches,Original_Comments,Reference
0,J000001.1-350334.9,MS2357.4-3520,NaN,0.00471,-35.05971,00:00:01.13-35:03:34.94,NaN,0.508,NaN,NS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Gioia et al. (1990) - doi:10.1086/191426
1,J000001.2+051908.6,NSCS J000001+051909,NaN,0.00504,5.31906,00:00:01.21+05:19:08.62,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Lopes et al. (2004) - doi:10.1086/423038
2,J000002.2+085518.4,NSCS J000002+085519,NaN,0.00925,8.92179,00:00:02.22+08:55:18.44,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Lopes et al. (2004) - doi:10.1086/423038
3,J000006.0+081629.7,NSCS J000006+081630,NaN,0.02512,8.27492,00:00:06.03+08:16:29.71,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Lopes et al. (2004) - doi:10.1086/423038
4,J000006.1+213829.4,DESI-000.0254+21.6415,NaN,0.02542,21.64150,00:00:06.10+21:38:29.40,C,0.502973,NaN,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Huang et al. (2021) - arXiv:2005.04730


In [3]:
Consolidated_Catalog_object = client.get_object(
    "slcomp", "Data/Consolidated_Data.csv"
).data
Database_Consolidated = pd.read_csv(
    io.StringIO(Consolidated_Catalog_object.decode("utf-8")),
    low_memory=False,
    dtype=object,
)
Database_Consolidated["RA"] = Database_Consolidated["RA"].astype(float)
Database_Consolidated["DEC"] = Database_Consolidated["DEC"].astype(float)
Database_Consolidated["mag_i"] = Database_Consolidated["mag_i"].astype(float)
Database_Consolidated.head()

,JNAME,RA,DEC,System_Type,Lens_Type,Source_Type,theta_E,theta_EErr,theta_EMethod,theta_ERef,...,mag_zS,mag_zSRef,mag_y,mag_yErr,mag_yRef,mag_F814W,mag_F814WErr,mag_F814WRef,mag_F814WS,mag_F814WSRef
0,J000001.1-350334.9,0.00471,-35.05971,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,J000001.2+051908.6,0.00504,5.31906,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,J000002.2+085518.4,0.00925,8.92179,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,J000006.0+081629.7,0.02512,8.27492,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,J000006.1+213829.4,0.02542,21.64150,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Select objects

Since SOAR telescope is located in the same place as Gemini-S telescope, we can use [their](https://www.gemini.edu/observing/phase-i/standard-semester-program/2021a-call-proposals/semester-2021a-instrument#South) information to restrict the objects:

- $6\,\mathrm{h}\,\leq\mathrm{RA}\leq20\,\mathrm{h}$
- $-75^{\circ}\leq\mathrm{DEC}\leq15^{\circ}$

In [4]:
Database = Database.query(
    "(RA>=6*15 and RA <=8*20) and (DEC >=-75 and DEC <= 15)"
).reset_index(drop=True)

In [5]:
len(Database)

2070

### How many of these objects has any match with `Consolidated Catalog`

In [6]:
data_with_match = Database_Consolidated[
    Database_Consolidated.JNAME.isin(Database.JNAME)
].reset_index(drop=True)
len(data_with_match)

2066

In [7]:
data_without_match = Database[~Database.JNAME.isin(data_with_match.JNAME)].reset_index(
    drop=True
)
len(data_without_match)

4

This means that only 4 objects doesn't have any match with our consolidated catalog. This is fine for us right now.

### Find systems with known $z_L$ and $z_S$

In [8]:
data_with_match = data_with_match[
    ~data_with_match.z_S.isna() & ~data_with_match.z_L.isna()
].reset_index(drop=True)
len(data_with_match)

297

### Obtain only bright systems, let's say those with $\mathrm{mag_i}<19$ (if available)

In [9]:
len(
    data_with_match[
        (data_with_match.mag_i <= 19) | (data_with_match.mag_i.isna())
    ].reset_index(drop=True)
)

146

In [10]:
data_with_match = data_with_match[
    (data_with_match.mag_i <= 19) | (data_with_match.mag_i.isna())
].reset_index(drop=True)

### Objects at individual scale, isolated

In [11]:
data_with_match = data_with_match[
    data_with_match.System_Type.isin([np.nan, "Single Lens Galaxy"])
].reset_index(drop=True)
len(data_with_match)

119

### Search for Cutouts

In [12]:
Cutouts_object_content = client.get_object(
    "slcomp", "Cutouts/Processed_Cutouts.parquet"
).data
Cutouts = pd.read_parquet(io.BytesIO(Cutouts_object_content))
Cutouts.head()

,JNAME,survey,cutout_size,processing,is_rgb,file_name,file_path
0,J000001.1-350334.9,DES,20asec,lsb,True,J000001.1-350334.9.jpeg,Processed_Cutouts_Legacy/20asec/DES/J000001.1-...
1,J000001.1-350334.9,Legacy,20asec,trilogy,False,J000001.1-350334.9_legacysurvey-0001m350-image...,Processed_Cutouts/20asec/Legacy/J000001.1-3503...
2,J000001.1-350334.9,Legacy,20asec,trilogy,False,J000001.1-350334.9_legacysurvey-0001m350-image...,Processed_Cutouts/20asec/Legacy/J000001.1-3503...
3,J000001.1-350334.9,Legacy,20asec,lsb,True,J000001.1-350334.9.jpeg,Processed_Cutouts_Legacy/20asec/Legacy/J000001...
4,J000001.1-350334.9,DES,4amin,lsb,True,J000001.1-350334.9.jpeg,Processed_Cutouts_Legacy/4amin/DES/J000001.1-3...


In [13]:
query_data = (
    Cutouts[Cutouts.JNAME.isin(data_with_match.JNAME)]
    .reset_index(drop=True)
    .query('cutout_size == "20asec" and is_rgb == "True"')
    .reset_index(drop=True)
)

In [14]:
query_data.JNAME.unique().shape[0], data_with_match.JNAME.unique().shape[0]

(119, 119)

In [15]:
rgb_cutouts = query_data[["JNAME", "survey", "file_path"]]
rgb_cutouts.iloc[0:3]

,JNAME,survey,file_path
0,J062736.3-542657.7,DES,Processed_Cutouts/20asec/DES/J062736.3-542657....
1,J062736.3-542657.7,DES,Processed_Cutouts_Legacy/20asec/DES/J062736.3-...
2,J062736.3-542657.7,Legacy,Processed_Cutouts/20asec/Legacy/J062736.3-5426...


We then create mosaics for these objects which it will be used later for manual visual inspection:

In [16]:
for jname in rgb_cutouts.JNAME.unique():
    data = rgb_cutouts[rgb_cutouts.JNAME == jname].reset_index(drop=True)

    nrow = 1
    ncol = len(data)

    fig = plt.figure(figsize=(ncol + 1, nrow + 1))

    gs = gridspec.GridSpec(
        nrow,
        ncol,
        wspace=0.0,
        hspace=0.0,
        top=1.0 - 0.5 / (nrow + 1),
        bottom=0.5 / (nrow + 1),
        left=0.5 / (ncol + 1),
        right=1 - 0.5 / (ncol + 1),
    )

    k = 0
    for i in range(nrow):
        for j in range(ncol):
            data_io = io.BytesIO(
                client.get_object("slcomp", "Cutouts/" + data.iloc[k].file_path).data
            )
            # im = plt.imread(data_io)
            im = Image.open(data_io)
            im.load()
            ax = plt.subplot(gs[i, j])
            if data.iloc[k].survey == "CS82":
                ax.imshow(im, interpolation="lanczos", aspect="auto", cmap="gray")
            else:
                ax.imshow(im, interpolation="lanczos", aspect="auto")
            ax.set_xticklabels([])
            ax.set_yticklabels([])
            ax.set_xticks([])
            ax.set_yticks([])

            left, width = 0.05, 0.5
            bottom, height = 0.05, 0.5

            ax.text(
                left,
                bottom,
                f"{data.iloc[k].survey}",
                horizontalalignment="left",
                verticalalignment="bottom",
                transform=ax.transAxes,
                color="gold",
                fontsize=10,
            )
            k += 1

    # plt.show()
    os.makedirs(name="./mosaics", exist_ok=True)

    plt.savefig(f"mosaics/{jname}.png", dpi=150, transparent=True)
    plt.close()

    im = Image.open(f"mosaics/{jname}.png")
    im2 = im.crop(im.getbbox())
    im2.save(f"mosaics/{jname}.png")

### Merge RGB cutouts with Consolidated Data

In [17]:
data_with_match_with_rgb = data_with_match[
    data_with_match.JNAME.isin(rgb_cutouts.JNAME.unique())
].reset_index(drop=True)
len(data_with_match_with_rgb)

119

We will add this data later on a shareable Google sheets. Also, we add the mosaics for each visualization of the mosaics:

In [18]:
data_with_match_with_rgb.insert(
    0,
    "IMAGE",
    [
        f'=IMAGE("https://raw.githubusercontent.com/oliveirara/slcomp-proposals/main/mosaics/{jname}.png")'
        for jname in data_with_match_with_rgb.JNAME
    ],
)

In [19]:
data_with_match_with_rgb.to_csv("proposals.csv", index=False)

We added all mosaics [here](https://github.com/oliveirara/slcomp-proposals) and import the `csv` file into this [spreadsheet](https://docs.google.com/spreadsheets/d/e/2PACX-1vQhpKpIYm_-k7zxWJM1OPAGslm4GMr2O1Hogqz3qyCV6_heqtXTOmScxlcuuWRSqxgTa0doIXCQZvtQ/pubhtml?gid=0&single=true).

### Visibility Plots

For visibility plots of selected objects, we recommend using [Object Visibility Plotter](https://astro.subhashbose.com/tools/object-visibility-plotter).